In [1]:
import torch
import torch.nn as nn

class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = torch.randn(3, requires_grad=True)   # nn.Parameter 아님
        self.b = nn.Parameter(torch.zeros(3))

m = M()
loss = ((m.w * 2 + m.b) ** 2).sum()
loss.backward()

print(len(list(m.parameters())), m.w.grad is None, m.b.grad is None)

1 False False


In [2]:
m = M()
((m.w*2 + m.b)**2).sum().backward()
print("1회:", m.w.grad, m.b.grad)

m.zero_grad()
print("zero_grad 후:", m.w.grad, m.b.grad)

((m.w*2 + m.b)**2).sum().backward()
print("2회:", m.w.grad, m.b.grad)

1회: tensor([-1.9099,  0.8700, -1.9028]) tensor([-0.9549,  0.4350, -0.9514])
zero_grad 후: tensor([-1.9099,  0.8700, -1.9028]) None
2회: tensor([-3.8197,  1.7401, -3.8057]) tensor([-0.9549,  0.4350, -0.9514])


In [3]:
model = nn.Linear(4, 3)     # 특징 4개 → 클래스 3개
x = torch.randn(2, 4)       # 샘플 2개
logits = model(x)

print(logits)               # 값의 범위에 제한이 없다
print(logits.shape)         # torch.Size([2, 3])  = (샘플 수, 클래스 수)
print(logits.sum(dim=1))    # 1이 아니다

tensor([[ 0.4756, -0.7308,  0.9030],
        [ 0.4146, -0.4713,  0.5676]], grad_fn=<AddmmBackward0>)
torch.Size([2, 3])
tensor([0.6477, 0.5109], grad_fn=<SumBackward1>)


# S1

In [4]:
import torch
import torch.nn as nn

pred = torch.tensor([2.0, 4.0, 6.0])
y    = torch.tensor([1.0, 4.0, 9.0])

print("손으로 :", ((pred - y) ** 2).mean().item())
print("mean   :", nn.MSELoss()(pred, y).item())
print("sum    :", nn.MSELoss(reduction='sum')(pred, y).item())
print("none   :", nn.MSELoss(reduction='none')(pred, y))

손으로 : 3.3333332538604736
mean   : 3.3333332538604736
sum    : 10.0
none   : tensor([1., 0., 9.])


In [5]:
torch.manual_seed(0)
x = torch.linspace(-3, 3, 100).view(-1, 1)
y = 3 * x + 2

model = nn.Linear(1, 1)
criterion = nn.MSELoss()

for i in range(100):
    pred = model(x)
    loss = criterion(pred, y)          # ← 손으로 쓰던 ((pred - y) ** 2).mean()
    model.zero_grad()
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= 0.1 * p.grad

print(model.weight.item(), model.bias.item())   # 3.0, 2.0 근처

3.0 1.999999761581421


-  한 줄로 적기 — 여기서 reduction='sum'으로 바꾸면 같은 lr=0.1에서 왜 학습이 터지는지.

> 오차 값을 모두 더하므로 차원을 거듭할수록 값이 커져서 결국 발산한다
>

# S2

In [6]:
z = torch.tensor([2.0, 1.0, 0.1])

p1 = torch.softmax(z, dim=0)
p2 = torch.softmax(z + 100, dim=0)

print(p1)
print(p2)
print(torch.allclose(p1, p2))

tensor([0.6590, 0.2424, 0.0986])
tensor([0.6590, 0.2424, 0.0986])
True


내 예측 : A
- 66 33 0
- 66 33 0
- True


In [7]:
print(torch.exp(z))         # [7.39, 2.72, 1.11]
print(torch.exp(z + 100))   # ???

tensor([7.3891, 2.7183, 1.1052])
tensor([inf, inf, inf])


In [8]:
Z = torch.tensor([[2.0, 1.0, 0.1],
                  [0.5, 2.5, 1.0]])      # 샘플 2개, 클래스 3개

print(torch.softmax(Z, dim=0).sum(dim=0))
print(torch.softmax(Z, dim=1).sum(dim=1))

tensor([1.0000, 1.0000, 1.0000])
tensor([1.0000, 1.0000])


⟶ 한 줄로 적기 — dim=0으로 잘못 썼을 때 1이 되는 건 무엇인지.
- 축을 정하는거구나. 축이 잘못됐으니 값이 이상하게 나오는거야. 근데 오류는 왜 안나는지 모르겠어.

# S3

In [9]:
z = torch.tensor([[2.0, 1.0, 0.1]])     # 샘플 1개, 클래스 3개
t = torch.tensor([0])                   # 정답은 0번

ce = nn.CrossEntropyLoss()

a = ce(z, t)                            # 로짓 그대로
b = ce(torch.softmax(z, dim=1), t)      # softmax를 먼저 씌워서

print(a.item(), b.item())

0.4170299470424652 0.8021209239959717


- 0.4170이 나오는 쪽은 어디이고, 다른 쪽은 어떻게 될까요?
- D. 둘 다 0.4170 — CrossEntropyLoss가 알아서 처리한다

In [10]:
import torch.nn.functional as F

logp = F.log_softmax(z, dim=1)
print(logp)                    # log(확률) 세 개

print(F.nll_loss(logp, t))     # 정답 칸을 골라 부호를 뒤집는다
print(ce(z, t))                # 같은 값

tensor([[-0.4170, -1.4170, -2.3170]])
tensor(0.4170)
tensor(0.4170)


In [11]:
torch.manual_seed(0)
z4 = torch.randn(4, 3)                  # (N, C)
t4 = torch.tensor([0, 2, 1, 1])         # (N,)  dtype은 long

print(ce(z4, t4))

# 손계산으로 대조
p4 = torch.softmax(z4, dim=1)
manual = -torch.log(p4[torch.arange(4), t4]).mean()
print(manual)

tensor(1.1304)
tensor(1.1304)


⟶ 한 줄로 적기 — nll_loss가 실제로 하는 일을 자기 말로.
- 부호 뒤집기. 왜 뒤집냐? 그건 몰라

# S4

In [12]:
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
t = torch.tensor([0])

loss = nn.CrossEntropyLoss()(z, t)
loss.backward()

print("실측 :", z.grad)

p = torch.softmax(z.detach(), dim=1)
onehot = torch.zeros_like(p)
onehot[0, t] = 1.0
print("식(5):", p - onehot)

실측 : tensor([[-0.3410,  0.2424,  0.0986]])
식(5): tensor([[-0.3410,  0.2424,  0.0986]])


In [13]:
z_bad = torch.tensor([[-10.0, 10.0]], requires_grad=True)   # 정답은 0번인데 1번에 몰빵
t2 = torch.tensor([0])

nn.CrossEntropyLoss()(z_bad, t2).backward()
print("CE  :", z_bad.grad)

z_bad.grad = None
p = torch.softmax(z_bad, dim=1)
target_onehot = torch.tensor([[1.0, 0.0]])
nn.MSELoss()(p, target_onehot).backward()
print("MSE :", z_bad.grad)

CE  : tensor([[-1.,  1.]])
MSE : tensor([[-4.1223e-09,  0.0000e+00]])


⟶ 한 줄로 적기 — MSE 쪽 기울기가 0에 가까운 이유를 자기 말로.
- 몰러

# S5


In [14]:
z_big = torch.tensor([[1000.0, 999.0, 998.0]])
t = torch.tensor([0])

p_naive = torch.exp(z_big) / torch.exp(z_big).sum()     # 식 (1) 그대로
print("직접    :", p_naive)
print("직접 CE :", -torch.log(p_naive[0, 0]))

print("torch   :", torch.softmax(z_big, dim=1))
print("torch CE:", nn.CrossEntropyLoss()(z_big, t))

직접    : tensor([[nan, nan, nan]])
직접 CE : tensor(nan)
torch   : tensor([[0.6652, 0.2447, 0.0900]])
torch CE: tensor(0.4076)


In [15]:
z_shift = z_big - z_big.max()
p_safe = torch.exp(z_shift) / torch.exp(z_shift).sum()

print(p_safe)
print(torch.softmax(z_big, dim=1))
print(torch.allclose(p_safe, torch.softmax(z_big, dim=1)))

tensor([[0.6652, 0.2447, 0.0900]])
tensor([[0.6652, 0.2447, 0.0900]])
True


⟶ 한 줄로 적기 — 최댓값을 빼도 결과가 같은 근거가 몇 번 식인지, 그리고 왜인지.
- 6. 1에서 확률을 빼니까 당연히 값은 같은 거 아님?

# C3

In [16]:
import torch
import torch.nn as nn

torch.manual_seed(0)
x = torch.randn(256, 8)
t = torch.randint(0, 10, (256,))      # 클래스 10개

model = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32, 10),
    # nn.Softmax(dim=1),
)
criterion = nn.CrossEntropyLoss()

for i in range(300):
    loss = criterion(model(x), t)
    model.zero_grad()
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= 0.5 * p.grad
    if i % 60 == 0:
        print(i, round(loss.item(), 4))

# 어디를 고쳐야 합니까? loss = criterion(model(x), t)

# 진단 근거 한 줄 — 어느 출력의 어떤 점을 보고 그렇게 판단했습니까? (개념 D의 표를 다시 보세요. 그리고 model(x)를 한 줄 찍어보면 확정됩니다)
# 아래 코드를 보면 model(x)는 확률인데 로짓(실제 값)을 넘겨야 함
# nn.Softmax(dim=1), 를 주석처리하면 잘 됨 softmax에서 확률로 바꿔버리니까 그 전 로짓값을 받으면 됨

# x → Linear(8,32) → ReLU → Linear(32,10) → Softmax(dim=1) → model(x)
#                              ↑                   ↑
#                          여기가 로짓        여기서 확률로 바뀜

0 2.3495
60 1.9872
120 1.7684
180 1.5549
240 1.3738


In [17]:
print(model(x)[0])
print(model(x)[0].sum())

tensor([ 2.6960,  0.4840, -1.3635,  0.7372, -0.8666, -1.8170, -1.0664, -1.6589,
         0.2527,  1.2500], grad_fn=<SelectBackward0>)
tensor(-1.3524, grad_fn=<SumBackward0>)


In [29]:
torch.manual_seed(0)
x = torch.linspace(-3, 3, 100).view(-1, 1)   # (100, 1)
y = (3 * x + 2).view(-1, 1)                     # (100,)

print((pred - y).shape)
print(x.shape, y.shape)

model = nn.Linear(1, 1)
criterion = nn.MSELoss()

for i in range(100):
    pred = model(x)                          # (100, 1)
    loss = criterion(pred, y)
    model.zero_grad()
    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= 0.1 * p.grad
    if i % 25 == 0:
        print(i, round(loss.item(), 4))

print(model.weight.item(), model.bias.item())


# 어디를 고쳐야 합니까? x y를 같은 차원으로 만들어줘야 하네

# 진단 근거 한 줄 — 무엇을 한 줄 찍어보면 확정할 수 있습니까? (경고 메시지를 끝까지 읽어 보세요)

# C:\Users\SBK\Desktop\own\LLMStudy\.venv\Lib\site-packages\torch\nn\modules\loss.py:630: UserWarning: Using a target size (torch.Size([100])) that is different to the input size (torch.Size([100, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
#  return F.mse_loss(input, target, reduction=self.reduction)

torch.Size([100, 1])
torch.Size([100, 1]) torch.Size([100, 1])
0 29.8251
25 0.0
50 0.0
75 0.0
3.0 1.999999761581421
